# MEDDIAG — DenseNet-121 full pipeline (Stage-1 + Stage-2 + eval)

Trains the DenseNet-121 model from scratch and evaluates it. Uploads to `vlm-densenet`. EfficientNet-B0 is fully removed; the encoder is TorchXRayVision DenseNet-121.


In [ ]:
# 1. CONFIG (DenseNet-121 — FULL pipeline: Stage-1 + Stage-2 + eval) ==========
# Trains the DenseNet-121 model FROM SCRATCH: Stage-1 (projector, 1024-dim) ->
# Stage-2 (LoRA + cls_head) -> eval -> 9 experiments. Uploads to vlm-densenet.
# The encoder is DenseNet-121 by default (src/config.py).
import os, re

IS_FIRST_SESSION = False          # RESUME: continue Stage-2 from the latest checkpoint
MODE = "train"                    # runs the full pipeline via run_pipeline.sh

# Practices carried forward: augmentation ON (OOD robustness); encoder FROZEN
# (preserve DenseNet-121's cross-dataset CXR features — fine-tuning would let it
# forget the other domains and hurt OOD). Standard loss for a clean baseline
# (raise CLS_POS_WEIGHT / CLS_FOCAL_GAMMA later if recall needs it). Eval applies
# SWA + Platt calibration + Youden threshold + TTA automatically.
VISION_AUGMENT   = True
VISION_FINETUNE  = False
CLS_POS_WEIGHT   = 1.0
CLS_FOCAL_GAMMA  = 0.0

def _detect_kaggle_username():
    for root, _dirs, _files in os.walk("/kaggle/input"):
        m = re.search(r"/kaggle/input/datasets/([^/]+)/", root.rstrip("/") + "/")
        if m:
            return m.group(1)
    return None

# Account for dataset upload + resume-download. PINNED to sujalprasad because
# sujalprasad0206 hit its dataset-storage quota (uploads 400'd on CreateDatasetVersion).
# Revert to `_detect_kaggle_username() or "sujalprasad"` once 0206 has room again.
KAGGLE_USERNAME = "sujalprasad"   # was: _detect_kaggle_username() or "sujalprasad"
STATE_DATASET   = "vlm-densenet"   # sujalprasad dataset (0206 quota full); attach it as INPUT to resume
EVAL_SAMPLES    = 200

MAX_PAIRS  = 4000
EPOCHS     = 3
GRAD_ACCUM = 4
SAVE_EVERY = 250
LR         = 2e-4

print("Config (DenseNet-121 full pipeline):")
print(f"  session        : {'FIRST (from scratch)' if IS_FIRST_SESSION else 'RESUME'}")
print(f"  kaggle user    : {KAGGLE_USERNAME}")
print(f"  augment        : {VISION_AUGMENT} | encoder fine-tune : {VISION_FINETUNE} (frozen = keep CXR features)")
print(f"  cls_pos_weight : {CLS_POS_WEIGHT} | focal_gamma : {CLS_FOCAL_GAMMA}")
print(f"  upload target  : {STATE_DATASET}")
print(f"  grad_accum     : {GRAD_ACCUM} | max_pairs : {MAX_PAIRS} | epochs : {EPOCHS}")


In [ ]:
# 2. PACKAGES =================================================================
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("--upgrade", "kaggle")          # ensure latest CLI (older one had an upload bug)
pip("peft>=0.19.1")
pip("bitsandbytes>=0.49.2")
pip("accelerate>=1.13.0")
pip("faiss-cpu==1.13.2")
pip("sentence-transformers")
pip("bert-score>=0.3.13")
pip("torchxrayvision>=1.2.0")   # DenseNet-121 CXR encoder           # BERTScore metric for exp2/exp3 (not in Kaggle base image)
pip("python-dotenv")
print("Packages ready.")


In [ ]:
# 3. SECRETS & ENV ============================================================
import os, torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"]                 = secrets.get_secret("HF_TOKEN")
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = secrets.get_secret("SEMANTIC_SCHOLAR_API_KEY")
os.environ["CUBLAS_WORKSPACE_CONFIG"]  = ":4096:8"
os.environ["MEDDIAG_MAX_VRAM_GB"]      = "14"   # full 16GB T4 (4GB hypothesis confirmed)
os.environ["MEDDIAG_MIMIC_REPO"]       = "power2004/mimic-cxr-dataset"   # replaces deleted itsanmolgupta mirror (same image+findings+impression, 30633 ex)

os.environ["MEDDIAG_VISION_AUGMENT"]  = "1" if VISION_AUGMENT else "0"
os.environ["MEDDIAG_VISION_FINETUNE"] = "1" if VISION_FINETUNE else "0"
os.environ["MEDDIAG_CLS_POS_WEIGHT"]   = str(CLS_POS_WEIGHT)
os.environ["MEDDIAG_CLS_FOCAL_GAMMA"]  = str(CLS_FOCAL_GAMMA)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Secrets loaded, HF login OK. Full 16 GB mode.")


In [ ]:
# 4. CLONE REPO (fresh) =======================================================
import subprocess, os
REPO_URL = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_DIR = "/kaggle/working/vlm"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, REPO_DIR])
    print("Cloned ->", REPO_DIR)
else:
    subprocess.call(["git", "-C", REPO_DIR, "reset", "--hard"])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print("Updated ->", REPO_DIR)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())


In [ ]:
# 5. RESTORE ASSETS ===========================================================
# Session 1 (IS_FIRST_SESSION=True): wipe models, restore only FAISS -> Stage-1
#   builds a fresh 1024-dim DenseNet projector.
# RESUME (False): restore projector + checkpoints + cls_head + FAISS from the
#   attached vlm-densenet INPUT MOUNT (/kaggle/input, read-only) if present, else
#   API-download the dataset. Handles both plain dirs and --dir-mode-zip archives.
#   NOTE: /kaggle/input is READ-ONLY. Training writes to /kaggle/working/vlm/models
#   and the uploader pushes those to sujalprasad/vlm-densenet — nothing is ever
#   saved into /kaggle/input.
import shutil, os, re, json, subprocess, zipfile
from pathlib import Path

MODELS_DIR = Path(REPO_DIR) / "models"; MODELS_DIR.mkdir(exist_ok=True)

def _walk_find(base, name, want_dir=False):
    for root, dirs, files in os.walk(base):
        if name in (dirs if want_dir else files):
            return Path(root) / name
    return None

def _restore_faiss():
    dst = Path(REPO_DIR) / "faiss_index"
    fa_dir = _walk_find("/kaggle/input", "faiss_index", want_dir=True)
    fa_zip = _walk_find("/kaggle/input", "faiss_index.zip", want_dir=False)
    idx    = _walk_find("/kaggle/input", "index.faiss", want_dir=False)
    if fa_dir:
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(fa_dir, dst); print(f"faiss_index restored (dir) from {fa_dir}")
    elif fa_zip:
        if dst.exists(): shutil.rmtree(dst)
        dst.mkdir(parents=True)
        with zipfile.ZipFile(fa_zip) as z: z.extractall(dst)
        nested = dst / "faiss_index"
        if nested.is_dir():
            for p in nested.iterdir(): shutil.move(str(p), str(dst / p.name))
            nested.rmdir()
        print(f"faiss_index restored (unzipped) from {fa_zip}")
    elif idx:
        # Kaggle flattens a single-folder upload -> index.faiss + meta.jsonl at root
        if dst.exists(): shutil.rmtree(dst)
        dst.mkdir(parents=True)
        shutil.copy2(idx, dst / "index.faiss")
        meta = idx.parent / "meta.jsonl"
        if meta.exists(): shutil.copy2(meta, dst / "meta.jsonl")
        print(f"faiss_index restored (loose files) from {idx.parent}")
    else:
        print("NOTE: no faiss_index in attached inputs — Stage-1 (step1) will BUILD it (slow).")

def _restore_bundle(base, models_dir, faiss_dst):
    """Restore projector + lora_step* + cls_head + faiss from `base` (an attached
    input mount like /kaggle/input/.../vlm-densenet, OR an API-download dir). Handles
    plain dirs AND Kaggle --dir-mode-zip archives (lora_step*.zip / faiss_index.zip),
    flat or single-folder-nested. Returns dict of what was found."""
    base = Path(base); models_dir = Path(models_dir); faiss_dst = Path(faiss_dst)
    models_dir.mkdir(parents=True, exist_ok=True)
    def _find(name, want_dir=False):
        for root, dirs, files in os.walk(base):
            if name in (dirs if want_dir else files): return Path(root) / name
        return None
    got = {"projector": False, "cls_head": False, "ckpts": [], "faiss": False}
    pj = _find("projector_stage1.pt")
    if pj: shutil.copy2(pj, models_dir / "projector_stage1.pt"); got["projector"] = True
    ch = _find("cls_head.pt")
    if ch: shutil.copy2(ch, models_dir / "cls_head.pt"); got["cls_head"] = True
    pat = re.compile(r"lora_step(\d+)$"); zpat = re.compile(r"(lora_step\d+)\.zip$")
    for root, dirs, files in os.walk(base):
        for d in list(dirs):
            if pat.match(d):
                dst = models_dir / d
                if dst.exists(): shutil.rmtree(dst)
                shutil.copytree(Path(root) / d, dst); got["ckpts"].append(d)
        for fn in files:
            m = zpat.match(fn)
            if m:
                name = m.group(1); dst = models_dir / name
                if dst.exists(): shutil.rmtree(dst)
                dst.mkdir(parents=True)
                with zipfile.ZipFile(Path(root) / fn) as z: z.extractall(dst)
                nested = dst / name
                if nested.is_dir():
                    for p in nested.iterdir(): shutil.move(str(p), str(dst / p.name))
                    nested.rmdir()
                got["ckpts"].append(name)
    fdir = _find("faiss_index", want_dir=True); fzip = _find("faiss_index.zip"); idx = _find("index.faiss")
    if fdir:
        if faiss_dst.exists(): shutil.rmtree(faiss_dst)
        shutil.copytree(fdir, faiss_dst); got["faiss"] = True
    elif fzip:
        if faiss_dst.exists(): shutil.rmtree(faiss_dst)
        faiss_dst.mkdir(parents=True)
        with zipfile.ZipFile(fzip) as z: z.extractall(faiss_dst)
        nested = faiss_dst / "faiss_index"
        if nested.is_dir():
            for p in nested.iterdir(): shutil.move(str(p), str(faiss_dst / p.name))
            nested.rmdir()
        got["faiss"] = True
    elif idx:
        if faiss_dst.exists(): shutil.rmtree(faiss_dst)
        faiss_dst.mkdir(parents=True)
        shutil.copy2(idx, faiss_dst / "index.faiss")
        meta = idx.parent / "meta.jsonl"
        if meta.exists(): shutil.copy2(meta, faiss_dst / "meta.jsonl")
        got["faiss"] = True
    return got

# GUARD (footgun-proof): if IS_FIRST_SESSION would wipe a completed Stage-1 but
# vlm-densenet already holds projector_stage1.pt, auto-switch to RESUME so a
# forgotten flag can never destroy Stage-1. Set FORCE_FRESH=True to intentionally
# discard Stage-1 and retrain (e.g. after an architecture change).
FORCE_FRESH = False
if IS_FIRST_SESSION and not FORCE_FRESH:
    # a projector already on the attached input mount also means "don't wipe"
    if _walk_find("/kaggle/input", "projector_stage1.pt") is not None:
        print("GUARD: projector_stage1.pt found on the input mount -> forcing RESUME.")
        IS_FIRST_SESSION = False
    else:
        try:
            _tok = secrets.get_secret("KAGGLE_KEY").strip()
            os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
            _kd = Path.home() / ".kaggle"; _kd.mkdir(exist_ok=True)
            if _tok.startswith("KGAT_"):
                os.environ["KAGGLE_API_TOKEN"] = _tok
                _f = _kd / "access_token"; _f.write_text(_tok); _f.chmod(0o600)
            else:
                os.environ["KAGGLE_KEY"] = _tok
                _f = _kd / "kaggle.json"; _f.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": _tok})); _f.chmod(0o600)
            _r = subprocess.run(["kaggle", "datasets", "files", f"{KAGGLE_USERNAME}/{STATE_DATASET}"],
                                capture_output=True, text=True)
            if "projector_stage1.pt" in (_r.stdout or ""):
                print(f"GUARD: {KAGGLE_USERNAME}/{STATE_DATASET} already has projector_stage1.pt -> forcing RESUME (set FORCE_FRESH=True to override).")
                IS_FIRST_SESSION = False
            else:
                print(f"GUARD: no projector in {STATE_DATASET} -> genuine fresh start; wiping as intended.")
        except Exception as _e:
            print(f"GUARD check skipped ({_e}); proceeding with IS_FIRST_SESSION={IS_FIRST_SESSION}")

if IS_FIRST_SESSION:
    for d in list(MODELS_DIR.glob("lora_step*")):
        if d.is_dir(): shutil.rmtree(d)
    for name in ("lora_adapter", "lora_adapter_swa"):
        if (MODELS_DIR / name).exists(): shutil.rmtree(MODELS_DIR / name)
    for name in ("cls_head.pt", "projector_stage1.pt"):
        (MODELS_DIR / name).unlink(missing_ok=True)
    print("from scratch: cleared stale models -> Stage-1 builds a fresh 1024-dim projector")
    _restore_faiss()
else:
    faiss_dst = Path(REPO_DIR) / "faiss_index"
    # 1) Restore ONLY from the vlm-densenet mount (matched by name), so a co-attached
    #    B0 dataset (vlm-session-state, EfficientNet-B0 checkpoints 9750-15000) can
    #    NEVER be pulled in and contaminate the DenseNet resume. Unpacks zips; no API.
    _dn = [q for q in Path("/kaggle/input").rglob(STATE_DATASET) if q.is_dir()]
    _base = str(_dn[0]) if _dn else "/kaggle/input"
    print(f"restore source: {_base}" + ("" if _dn else f"  (no '{STATE_DATASET}' mount; arch-check still guards B0)"))
    got = _restore_bundle(_base, MODELS_DIR, faiss_dst)
    print(f"restored: projector={got['projector']} "
          f"cls_head={got['cls_head']} ckpts={sorted(got['ckpts'])} faiss={got['faiss']}")
    # 2) Fallback to API download only if the input mount lacked the projector.
    if not got["projector"]:
        print(f"RESUME: input mount had no projector -> downloading {KAGGLE_USERNAME}/{STATE_DATASET} ...")
        try:
            token = secrets.get_secret("KAGGLE_KEY").strip()
            os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
            kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
            if token.startswith("KGAT_"):
                os.environ["KAGGLE_API_TOKEN"] = token
                f = kdir / "access_token"; f.write_text(token); f.chmod(0o600)
            else:
                os.environ["KAGGLE_KEY"] = token
                f = kdir / "kaggle.json"; f.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": token})); f.chmod(0o600)
            DL = Path("/kaggle/working/_dn_dl")
            if DL.exists(): shutil.rmtree(DL)
            DL.mkdir(parents=True)
            r = subprocess.run(["kaggle", "datasets", "download", f"{KAGGLE_USERNAME}/{STATE_DATASET}",
                                "--unzip", "-p", str(DL)], capture_output=True, text=True)
            print((r.stdout or "")[-200:]); print((r.stderr or "")[-200:])
            got = _restore_bundle(DL, MODELS_DIR, faiss_dst)
            print(f"restored from download: projector={got['projector']} "
                  f"cls_head={got['cls_head']} ckpts={sorted(got['ckpts'])} faiss={got['faiss']}")
        except Exception as e:
            print(f"RESUME download failed ({e}); restoring FAISS from input — may restart Stage-1")
    # ensure faiss is present even if the bundle missed it
    if not (faiss_dst / "index.faiss").exists():
        _restore_faiss()

# HARD ARCH GUARD: the restored projector MUST be DenseNet-121 (1024-dim vision_proj
# input). EfficientNet-B0 is 1280-dim; if a B0 projector slipped in, ABORT here rather
# than silently training/evaluating a mismatched checkpoint. This is the definitive
# B0-vs-DenseNet discriminator (lora_step adapters are shape-identical across encoders).
_pjp = MODELS_DIR / "projector_stage1.pt"
if _pjp.exists():
    import torch as _torch
    _sd = _torch.load(_pjp, map_location="cpu")
    _vp = next((v for k, v in _sd.items() if k.endswith("vision_proj.weight")), None)
    if _vp is not None:
        _indim = int(_vp.shape[1])
        if _indim != 1024:
            raise RuntimeError(
                f"ARCH MISMATCH: projector_stage1.pt expects {_indim}-dim vision features, but "
                f"DenseNet-121 produces 1024. This is an EfficientNet-B0 (1280-dim) checkpoint. "
                f"Detach the B0 dataset (vlm-session-state) and re-restore.")
        print(f"[arch-check] projector vision_proj in-dim = {_indim} -> DenseNet-121 OK")

_dst = Path(REPO_DIR) / "faiss_index"
print("faiss present:", (_dst / "index.faiss").exists() and (_dst / "meta.jsonl").exists())
print("projector present:", (MODELS_DIR / "projector_stage1.pt").exists())
print("\nAssets ready.")


In [ ]:
# 5b. SMOKE TEST — validate the DenseNet path BEFORE the multi-hour run ========
# Loads the encoder, runs one forward + projector pass, checks shapes. If the
# torchxrayvision API or any dim is wrong, it fails HERE (seconds) — not hours in.
import torch
from PIL import Image
from src.config import CONFIG
from src.vision import VisionEncoder
from src.projector import PerceiverResampler

print("vision_model_id :", CONFIG.vision_model_id)
print("vision_hidden_dim:", CONFIG.vision_hidden_dim)
assert CONFIG.vision_model_id.startswith("xrv:"), "encoder is not DenseNet/torchxrayvision!"

ve = VisionEncoder(CONFIG)
px = ve.preprocess(Image.new("L", (256, 256), 128))
print("preprocessed    :", tuple(px.shape), px.dtype)          # expect (1, 1, 224, 224)
assert px.shape == (1, 1, 224, 224), f"bad preprocess shape {tuple(px.shape)}"

tok = ve(px)
print("vision tokens   :", tuple(tok.shape))                    # expect (1, 49, 1024)
assert tok.shape[-1] == CONFIG.vision_hidden_dim == 1024, "feature dim != 1024"

proj = PerceiverResampler(vision_dim=CONFIG.vision_hidden_dim, llm_dim=CONFIG.llm_hidden_dim,
                          num_latents=CONFIG.num_visual_tokens,
                          num_heads=CONFIG.projector_num_heads, num_layers=CONFIG.projector_num_layers)
out = proj(tok.float())
print("projector out   :", tuple(out.shape))                    # expect (1, 8, 3072)
assert out.shape == (1, CONFIG.num_visual_tokens, CONFIG.llm_hidden_dim), "projector output shape wrong"

del ve, proj, tok, out; torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("\n[OK] DenseNet path is consistent (encoder + preprocess + projector). Safe to train.")


In [ ]:
# 6. PIPELINE STATE — resume-aware ===========================================
# Mark step0 (deps) done always. Mark step1 (FAISS) done if the index is ready,
# and step2 (Stage-1) done if a projector already exists (resume) — so
# run_pipeline runs only what's left: Stage-1 (only if no projector) -> Stage-2
# (resumes from the latest checkpoint) -> eval -> experiments.
from pathlib import Path
_faiss_ok = (Path(REPO_DIR) / "faiss_index" / "index.faiss").exists()
_proj_ok  = (Path(REPO_DIR) / "models" / "projector_stage1.pt").exists()
sp = Path(REPO_DIR) / "logs" / ".pipeline_state"; sp.parent.mkdir(exist_ok=True)
markers = ["step0"] * 7 + (["step1"] if _faiss_ok else []) + (["step2"] if _proj_ok else [])
with open(sp, "w") as f:
    for m in markers:
        f.write(m + chr(10))
(Path(REPO_DIR) / "logs" / ".pipeline.lock").unlink(missing_ok=True)
print(f"Pipeline state: faiss={_faiss_ok}, projector={_proj_ok} -> "
      + ("RESUME Stage-2" if _proj_ok else "run Stage-1 (build projector) first"))


In [ ]:
# 7. PATCH run_pipeline.sh (grad_accum / max_pairs) ===========================
import re
from pathlib import Path
sh = Path(REPO_DIR) / "run_pipeline.sh"; t = sh.read_text()
t = re.sub(r"(GRAD_ACCUM=)\d+",   f"GRAD_ACCUM={GRAD_ACCUM}",  t)
t = re.sub(r"(MAX_PAIRS_S2=)\d+", f"MAX_PAIRS_S2={MAX_PAIRS}", t)
sh.write_text(t)
print(f"patched: GRAD_ACCUM={GRAD_ACCUM}, MAX_PAIRS_S2={MAX_PAIRS}")


In [ ]:
# 8. RUN — TRAIN or EVAL (set MODE in cell 1) =================================
# eval : SWA(last 8) -> calibrate -> evaluate(--tta --rag-ablation --robustness)
#        -> graphs -> upload results to the vlm-eval-results dataset. No training.
# train: resume Stage-2 to 15000, auto-uploading checkpoints to vlm-densenet.
import subprocess, os, shutil, re, json, glob, threading
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"   # live (unbuffered) child output -> real-time per-sample progress

MODELS = Path(REPO_DIR) / "models"
pat = re.compile(r"lora_step(\d+)$")
steps = sorted(int(pat.match(p.name).group(1)) for p in MODELS.iterdir() if pat.match(p.name))
print(f"checkpoints present: {steps}" if steps else "No checkpoints found!")
print("=" * 60)

def kaggle_auth():
    try:
        token = secrets.get_secret("KAGGLE_KEY").strip()
    except Exception as e:
        print(f"  !! KAGGLE_KEY secret missing ({e}); skipping upload."); return False
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
    if token.startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = token
        f = kdir / "access_token"; f.write_text(token); f.chmod(0o600)
    else:
        os.environ["KAGGLE_KEY"] = token
        f = kdir / "kaggle.json"; f.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": token})); f.chmod(0o600)
    return True

def _stream(cmd):
    print("\n>>> " + " ".join(cmd)); print("-" * 60)
    p = subprocess.Popen(cmd, cwd=REPO_DIR, env=os.environ.copy(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    return p.returncode

def _upload_dir(out_dir, slug, msg):
    if not kaggle_auth():
        print("  no KAGGLE_KEY — results left in", out_dir); return
    (Path(out_dir) / "dataset-metadata.json").write_text(json.dumps(
        {"title": slug, "id": f"{KAGGLE_USERNAME}/{slug}", "licenses": [{"name": "CC0-1.0"}]}, indent=2))
    r = subprocess.run(["kaggle", "datasets", "version", "-p", str(out_dir), "-m", msg, "--dir-mode", "zip"],
                       capture_output=True, text=True)
    blob = (r.stderr or "") + (r.stdout or "")
    if r.returncode != 0 and ("not found" in blob.lower() or "404" in blob):
        r = subprocess.run(["kaggle", "datasets", "create", "-p", str(out_dir), "--dir-mode", "zip"],
                           capture_output=True, text=True)
    print((r.stdout or "")[-400:]); print((r.stderr or "")[-200:])

_upload_lock = threading.Lock()
def package_and_upload(msg="auto-update"):
    """Train-mode: upload EVERY checkpoint (with train_state) to vlm-densenet."""
    with _upload_lock:
        OUT = Path("/kaggle/working/session_state")
        if OUT.exists(): shutil.rmtree(OUT)
        OUT.mkdir(parents=True)
        M = Path(REPO_DIR) / "models"; L = Path(REPO_DIR) / "logs"
        p = re.compile(r"lora_step(\d+)$")
        cps = sorted((int(p.match(x.name).group(1)), x) for x in M.iterdir() if p.match(x.name))
        for _, c in cps:
            dst = OUT / c.name
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(c, dst)
        s = M / "cls_head.pt"
        if s.exists(): shutil.copy2(s, OUT / "cls_head.pt")
        pj = M / "projector_stage1.pt"
        if pj.exists(): shutil.copy2(pj, OUT / "projector_stage1.pt")   # DenseNet projector
        fa = Path(REPO_DIR) / "faiss_index"
        if fa.exists(): shutil.copytree(fa, OUT / "faiss_index")
        lg = L / "stage2.jsonl"
        if lg.exists(): shutil.copy2(lg, OUT / "stage2.jsonl")
        # eval + experiment results (present after run_pipeline steps 4-6)
        for ej in L.glob("eval_report_*.json"):
            shutil.copy2(ej, OUT / ej.name)
        for sub in ("reports", "diagnostics"):
            srcd = Path(REPO_DIR) / sub
            if srcd.exists() and any(srcd.iterdir()):
                dsub = OUT / sub
                if dsub.exists(): shutil.rmtree(dsub)
                shutil.copytree(srcd, dsub)
        pj_ok = (M / "projector_stage1.pt").exists()
        if not cps and not pj_ok:
            print("  nothing to upload yet"); return False
        print(f"  packaged {len(cps)} checkpoints + projector={pj_ok} + eval results")
        _upload_dir(OUT, STATE_DATASET, msg); return True

# ════════════════════════════════════════════════════════════════════════════
if MODE == "eval":
    print(">>> EVAL MODE: SWA + calibrate + improved evaluation (no training)\n")
    rc = _stream(["python", "-m", "experiments.average_checkpoints",
                  "--models-dir", "models", "--last-n", "8", "--out", "models/lora_adapter_swa"])
    if rc == 0:
        rc = _stream(["python", "-m", "experiments.calibrate_temperature",
                      "--lora-dir", "models/lora_adapter_swa", "--val-pairs", "500"])
    if rc == 0:
        rc = _stream(["python", "-m", "experiments.evaluate",
                      "--lora-adapter-dir", "models/lora_adapter_swa",
                      "--max-samples", str(EVAL_SAMPLES), "--tta", "--rag-ablation", "--robustness"])
    if rc == 0:
        # FULL ablation study — all 9 experiments (Exp 1-9) on the SWA model
        rc = _stream(["python", "-m", "experiments.run_experiments", "--exp", "all",
                      "--projector", "models/projector_stage1.pt",
                      "--lora-adapter", "models/lora_adapter_swa",
                      "--max-samples", str(EVAL_SAMPLES), "--tgp-w", "55",
                      "--output-dir", "reports/"])
    if rc == 0:
        # single-image qualitative inference -> reports/single_inference.{json,md,png}
        _stream(["python", "-m", "experiments.single_inference",
                 "--image", "sample_xray.jpg",
                 "--lora-adapter-dir", "models/lora_adapter_swa",
                 "--projector-path", "models/projector_stage1.pt",
                 "--output-dir", "reports/"])
    if rc != 0:
        print("\n[eval] a step failed (rc != 0) — see output above; uploading whatever completed.")
    ej = sorted(glob.glob(str(Path(REPO_DIR) / "logs" / "eval_report_*.json")))
    if rc == 0 and ej:
        _stream(["python", "-m", "experiments.visualize", "--eval-json", ej[-1],
                 "--stage2-log", "logs/stage2.jsonl", "--out-dir", "diagnostics/", "--format", "png"])
    # stage + upload eval results (small) to vlm-eval-results
    OUT = Path("/kaggle/working/eval_results")
    if OUT.exists(): shutil.rmtree(OUT)
    OUT.mkdir(parents=True)
    if ej: shutil.copy2(ej[-1], OUT / Path(ej[-1]).name)
    diag = Path(REPO_DIR) / "diagnostics"
    if diag.exists() and any(diag.iterdir()): shutil.copytree(diag, OUT / "diagnostics")
    rep = Path(REPO_DIR) / "reports"
    if rep.exists() and any(rep.iterdir()): shutil.copytree(rep, OUT / "reports")   # Exp 1-9 results
    swa = Path(REPO_DIR) / "models" / "lora_adapter_swa"
    if swa.exists(): shutil.copytree(swa, OUT / "lora_adapter_swa")
    chp = Path(REPO_DIR) / "models" / "cls_head.pt"
    if chp.exists(): shutil.copy2(chp, OUT / "cls_head.pt")
    print("\n[eval] uploading results to vlm-eval-results ...")
    _upload_dir(OUT, "vlm-eval-results", "eval: SWA + calibrate + TTA")
    print("\nEVAL COMPLETE -> results in the vlm-eval-results dataset + /kaggle/working/eval_results/")
else:
    if steps:
        ok = (MODELS / f"lora_step{steps[-1]}" / "train_state.pt").exists()
        print(f"RESUME TARGET: lora_step{steps[-1]}  (train_state.pt: {'OK' if ok else 'MISSING!'})")
    _stop = threading.Event()
    def _periodic():
        last = -1; proj_up = False
        while not _stop.wait(900):
            cur = max([int(pat.match(p.name).group(1)) for p in MODELS.iterdir() if pat.match(p.name)] or [-1])
            proj_now = (MODELS / "projector_stage1.pt").exists()
            if cur > last or (proj_now and not proj_up):
                print(f"\n[uploader] checkpoint={cur} projector={proj_now} -> uploading ...")
                if package_and_upload(f"autosave (step {cur})"):
                    last = cur; proj_up = proj_now
    threading.Thread(target=_periodic, daemon=True).start()
    print("[uploader] auto-upload ON\n")
    proc = subprocess.Popen(["bash", "run_pipeline.sh", "--resume"], cwd=REPO_DIR,
                            env=os.environ.copy(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate(); print("\n[notebook] interrupted - saving now...")
    proc.wait(); _stop.set()
    print(f"\n[notebook] exit code {proc.returncode}")
    package_and_upload("final - all checkpoints")
    print("Download: https://www.kaggle.com/datasets/" + KAGGLE_USERNAME + "/" + STATE_DATASET)


In [ ]:
# 9. FORCE UPLOAD NOW (optional) — pushes ALL checkpoints immediately ==========
# Run any time you want to snapshot the full set right now (does not stop training
# only if cell 8 has finished/been interrupted, since the kernel is otherwise busy).
package_and_upload("manual full snapshot")


## Where outputs go (and the /kaggle/input read-only rule)

`/kaggle/input` is **read-only** — training cannot write there. Checkpoints are
written to `/kaggle/working/vlm/models`, and the uploader (cell 8) pushes them to
**`sujalprasad/vlm-densenet`**. To resume next session, **attach that dataset**: it
mounts read-only at `/kaggle/input/.../vlm-densenet` and cell 5 restores from it.

## Downloading the weights

You normally never need to — cell 8 pushes everything to `vlm-densenet`, and the
next session pulls it automatically.

To pull weights to your laptop (stable, unlike the in-session Output tab):
- **Dataset page**: https://www.kaggle.com/datasets/sujalprasad/vlm-densenet -> three-dot menu -> Download
- **CLI**:
  ```
  export KAGGLE_API_TOKEN=KGAT_xxx
  kaggle datasets download sujalprasad/vlm-densenet --unzip -p ./out
  ```

## What was fixed vs the old notebook
- No pruning anywhere — every checkpoint kept and uploaded.
- Resume verified before training (cell 8 prints the exact target step).
- `git reset --hard` before pull — no more merge conflicts.
- Auto-discover dataset paths — mount path can't break it.
- Auto-upload via `KGAT_` token — downloads actually work.
- Kermany pre-load capped at 400 (in repo) — no RAM OOM.
